# cross-entropy-classification-loss — worked example 3: Cross-entropy over a sequence by flattening time

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `cross-entropy-classification-loss`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import torch.nn.functional as F

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

In language modeling the logits are `(B, T, C)` and labels are `(B, T)`. `F.cross_entropy` expects `(N, C)` and `(N,)`, so you flatten the batch and time axes together into `N = B*T` before scoring. einops `rearrange` makes the flatten explicit and shape-safe.

## Worked solution

We score next-token predictions across a batch of sequences.

1. **Flatten predictions.** `rearrange(logits, 'b t c -> (b t) c')` merges batch and time into one axis, giving `(B*T, C)`. The class axis `c` stays last because cross-entropy classifies along it.
2. **Flatten labels to match.** `rearrange(labels, 'b t -> (b t)')` produces `(B*T,)` in the *same* row order as the flattened logits — this alignment is critical, otherwise tokens get scored against the wrong targets.
3. **Single cross-entropy call.** `F.cross_entropy(flat_logits, flat_labels)` averages the per-token NLL over all `B*T` positions, which is the standard LM loss.
4. **Why this equals the naive double loop.** Cross-entropy is a per-row independent computation followed by a mean; flattening just changes the iteration order, not the set of (logit-row, label) pairs, so the mean is unchanged.

The assert checks it against re-flattening manually with `.reshape`, confirming the einops version is correct.

In [ ]:
import torch.nn.functional as F
from einops import rearrange

def seq_cross_entropy(logits, labels):
    flat_logits = rearrange(logits, 'b t c -> (b t) c')
    flat_labels = rearrange(labels, 'b t -> (b t)')
    return F.cross_entropy(flat_logits, flat_labels)

t.manual_seed(0)
logits = t.randn(2, 4, 5)          # (B=2, T=4, C=5)
labels = t.randint(0, 5, (2, 4))
mine = seq_cross_entropy(logits, labels)
ref = F.cross_entropy(logits.reshape(-1, 5), labels.reshape(-1))
print("mine:", round(mine.item(), 6), "ref:", round(ref.item(), 6))
print("match:", t.allclose(mine, ref, atol=1e-5))